In [2]:
import weaviate
import weaviate.classes as wvc
from weaviate.classes.query import Filter
import pandas as pd
import numpy as np
import requests
from weaviate.classes.query import MetadataQuery
import sys
import argparse
import multiprocessing
from tqdm import tqdm

# Create argument parser
# parser = argparse.ArgumentParser(description='Process command line arguments')

# # Add arguments with default values
# parser.add_argument('--corpus', type=str, default='corpora/ft30k/test.tsv.gz', help='Corpus to be indexed')
# parser.add_argument('--index', type=str, default='corpora/ft30k/test.arrow', help='Index file')
# parser.add_argument('--alpha', type=float, default=0.0, help='Alpha value for hybrid search')
# parser.add_argument('--pref_labels', type=str, default='vocab/gnd_pref_labels.arrow', help='Data frame with preferred labels')
# parser.add_argument('--n_jobs', type=int, default=20, help='Number of parallel jobs')
# parser.add_argument('--output', type=str, default='results/test/predictions.arrow', help='Output file')

# # Parse arguments
# args = parser.parse_args()

# Access the arguments
corpus = 'corpora/ft30k/test.tsv.gz'
index = 'corpora/ft30k/test.arrow'
alpha = 0.0
if not FileExistsError(corpus):
  sys.exit("Corpus file does not exist. Exiting...")
pref_labels = 'vocab/gnd_pref_labels.arrow'
n_jobs = 20
output = 'results/test/predictions.arrow'

In [19]:
def index_chunk(text_query, doc_id, wv_collection, alpha, host='8090'):
    embedding_response = None
    for _ in range(100):
        try:
            embedding_response = requests.post(
                'http://127.0.0.1:{}/embed'.format(host),
                headers={"Content-Type": "application/json"},
                json={'inputs': text_query}).json()
            break
        except Exception as e:
            # print("An error occurred:", str(e))
            # print("Retrying...")
            time.sleep(1)
    
    if embedding_response is None:
        print("Failed to get embedding for text query:", text_query, "doc_id:", doc_id)
        return {}

    embedding = list(
        np.array(
            embedding_response
        ).reshape(-1)
    )

    response = wv_collection.query.hybrid(
        query=text_query,
        vector=embedding,
        limit=100,
        return_metadata=MetadataQuery(score=True),
        alpha=alpha
    )

    df = pd.DataFrame(
        [
            dict(
                doc_id=doc_id,
                label_id=o.properties['idn'],
                score=o.metadata.score
            )
            for o in response.objects
        ]
    )

    # group by label_id and only keep rows with the highest score per label_id
    if not df.empty:
        df = df.sort_values('score', ascending=False).groupby('label_id').head(1)
        return df
    else:
        return {}

def index_text(text: str, doc_id: str, wv_collection, alpha: float):
    # Split the text into chunks of 1000 characters
    chunks = [text[i:i + 1000] for i in range(0, len(text), 1000)]
    n_chunks = len(chunks)
    df = pd.DataFrame(columns=['doc_id', 'label_id', 'score'])
    for chunk in chunks:
        chunk_df = index_chunk(chunk, doc_id, wv_collection, alpha)
        if not chunk_df.empty:
            if not df.empty:
                df = pd.concat([df, chunk_df])
            else:
                df = chunk_df

    df = df.groupby('label_id').agg(
        score=('score', 'sum'),
        occurrences=('doc_id', 'count')
    ).reset_index()
    df.columns = ['label_id', 'score', 'occurrences']
    # normalize score by n_chunks
    df['score'] = df['score'] / n_chunks
    df['occurrences'] = df['occurrences'] / n_chunks
    return df

In [4]:
# Get the data from the tsv.gz corpus file
df_documents = pd.read_csv(
    corpus,
    compression="gzip",
    sep="\t",
    header=None,
    names=["content", "ground_truth"],
)

df_index = pd.read_feather(index)
df_documents["doc_id"] = df_index.idn

In [5]:
pref_labels = pd.read_feather(pref_labels)

In [3]:
client = weaviate.connect_to_local()
if not client.is_ready():
      sys.exit("Weaviate client is not ready. Exiting...")

In [5]:
client.close()

In [5]:
client.collections.delete("Title_test_baai_bge_m3")

In [7]:
wv_collection = client.collections.get('Gnd859k_baai_bge_m3')

In [31]:
res = wv_collection.query.fetch_objects(
        filters=(
            Filter.by_property('idn').equal('bla')
        ),
        limit=1, 
        include_vector = True
        )

In [33]:
if res.objects:
  print("object found")

In [27]:
if not res.objects:
    print("res.objects is empty")
else:
    print("res.objects is not empty")

res.objects is not empty


In [20]:
res.objects[0].vector['default']

IndexError: list index out of range

In [20]:
res = index_text(df_documents.content[1], df_documents.doc_id[1], wv_collection, alpha=1.0)
# left join with pref_labels on `label_id`
res = res.merge(pref_labels, on='label_id', how='left')
res = res.sort_values('score', ascending=False)
print(res.head(20))

       label_id     score  occurrences  \
755   043164129  0.284010     0.566667   
1054  96475617X  0.250869     0.366667   
887   941049167  0.211715     0.433333   
988   952602555  0.207803     0.466667   
488   041652436  0.185450     0.300000   
229   040490122  0.184670     0.633333   
512   041700686  0.182523     0.366667   
1079  972354840  0.182487     0.333333   
628   042089395  0.165198     0.300000   
495   041665686  0.153091     0.566667   
1109  985769424  0.147759     0.400000   
54    004615301  0.147581     0.366667   
90    004857666  0.133075     0.366667   
772   050213393  0.132805     0.333333   
124   007604130  0.127571     0.233333   
1074  970284667  0.126401     0.366667   
83    00485408X  0.124150     0.433333   
55    004617029  0.111339     0.233333   
126   00764468X  0.106456     0.233333   
12    000351741  0.104521     0.366667   

                                             label_text  
755                                   Landesbeteiligung  
1

In [15]:
df_documents.content[1]

'21 Die Landesministerkonferenzen und der Bund – Kooperativer Föderalismus im Schatten der Politikverfl echtung Yvonne Hegele / Nathalie Behnke The Conferences of the Ministers of the German ‘Länder’ and the Federal Level – Co-opera- tive Federalism in the shadow of ‘Politikverfl echtung’ Abstract: The Conferences of the Ministers of the German ‘Länder’ (LMKen) are classifi ed as horizontal co-ordination bodies in co-operative federalism, although the federal level is known to be involved in their negotiations. Our paper investigates conditions and extent of federal in volvement in order to come to a more precise understanding of the confer- ences’ role in German federalism. An analysis of resolutions of the LMK proceedings shows that federal involvement is far higher than theoretically expected. The LMKen serve not primarily as a shield against federal interference, but rather to coordinate and infl u- ence federal decision-making. The combination of obligatory negotiations (Politi

In [ ]:
# create a weaviate collection for chunked documents
client.collections.create(
    name="ft30k_test_baai_bge_m3",
    properties=[
        wvc.config.Property(
            name="doc_id",
            description="DNB internal identifier",
            data_type=wvc.config.DataType.TEXT,
            tokenization=wvc.config.Tokenization.WORD,
            vectorize_property_name=False,
            skip_vectorization=True,
            index_searchable=False,
            index_filterable=True,
        ),
        wvc.config.Property(
            name="chuck_id",
            description="Chunk identifier",
            data_type=wvc.config.DataType.INT,
            tokenization=None,
            vectorize_property_name=False,
            skip_vectorization=True,
            index_searchable=False,
            index_filterable=True,
        ),
        wvc.config.Property(
            name="chunk_text",
            description="Chunk text",
            data_type=wvc.config.DataType.TEXT,
            vectorize_property_name=False,
            tokenization=wvc.config.Tokenization.WORD,
            index_searchable=True,
            index_filterable=False,
        )
    ],
    vectorizer_config=None,
    # Configure the vector index
    vector_index_config=wvc.config.Configure.VectorIndex.hnsw(
        distance_metric=wvc.config.VectorDistances.COSINE
    )
)